In [1]:
import argparse
from collections import Counter
from pathlib import Path
 
import torch
from transformers import AutoTokenizer

/mmfs1/project/phan/tqn/.conda/envs/train_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def top_k_token_ids(avg_entropy: torch.Tensor, k: int) -> list[int]:
    """Return token IDs sorted by descending average entropy (seen tokens only)."""
    seen  = (avg_entropy > 0).nonzero(as_tuple=True)[0]
    vals  = avg_entropy[seen]
    order = vals.argsort(descending=True)[:k]
    return seen[order].tolist()
 
 
def load_trials(trials_dir: Path, glob: str) -> list[dict]:
    """Load all matching .pt files; return list of {path, entropy, ...}."""
    files = sorted(trials_dir.glob(glob))
    if not files:
        raise FileNotFoundError(f"No files matching '{glob}' found in {trials_dir}")
 
    trials = []
    for path in files:
        data = torch.load(path, map_location="cpu", weights_only=True)
        if "entropy" not in data:
            raise KeyError(f"{path} is missing the 'entropy' key.")
        trials.append({"path": path, **data})
        print(f"  Loaded {path.name}  (vocab_size={data['entropy'].shape[0]})")
    return trials

In [3]:
trials_dir = "../outputs/entropy"
trials_dir = Path(trials_dir)
glob = "*.pt"
min_count = 500
top_k = 20000
min_entropy=0.5

# ── load tokenizer (optional) ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B", trust_remote_code=True)

def tok_repr(tid: int) -> str:
    if tokenizer is None:
        return str(tid)
    return repr(tokenizer.convert_ids_to_tokens(tid))

# ── load trial files ──────────────────────────────────────────────────────
print(f"Loading trial files from: {trials_dir}  (glob='{glob}')")
trials = load_trials(trials_dir, glob)
n_trials = len(trials)
print(f"\n{n_trials} trial file(s) found.\n")

# ── apply min-count filter & collect top-k per trial ─────────────────────
trial_top_ids: list[set[int]] = []
trial_meta:    list[dict]     = []

for t in trials:
    entropy = t["entropy"].clone()

    if min_count > 0 and "count" in t:
        low_count = t["count"] < min_count
        entropy[low_count] = 0.0

    if min_entropy > 0 and "entropy" in t:
        low_entropy = t["entropy"] < min_entropy
        entropy[low_entropy] = 0.0


    top_ids = top_k_token_ids(entropy, top_k)
    trial_top_ids.append(set(top_ids))
    trial_meta.append({"path": t["path"], "entropy": entropy,
                        "count": t.get("count"), "top_ids": top_ids})

# ── per-trial summary ─────────────────────────────────────────────────────
for idx, meta in enumerate(trial_meta):
    print(f"{'─'*64}")
    print(f"Trial {idx+1}: {meta['path'].name}")
    print(f"  {'Rank':<6} {'Token ID':<12} {'Token repr':<30} {'Avg Entropy':>12}"
            + ("  Count" if meta["count"] is not None else ""))
    print(f"  {'─'*60}")
    for rank, tid in enumerate(meta["top_ids"], 1):
        ent = meta["entropy"][tid].item()
        cnt = meta["count"][tid].item() if meta["count"] is not None else ""
        print(f"  {rank:<6} {tid:<12} {tok_repr(tid):<30} {ent:>12.4f}"
                + (f"  {cnt}" if cnt != "" else ""))

# ── consistency analysis ──────────────────────────────────────────────────
print(f"\n{'='*64}")
print("CONSISTENCY ANALYSIS")
print(f"{'='*64}\n")

# 1. Universal intersection
universal = set.intersection(*trial_top_ids)
print(f"Tokens in top-{top_k} of ALL {n_trials} trials  →  {len(universal)} token(s):")
if universal:
    print(f"  {'Token ID':<12} {'Token repr':<30} {'Mean entropy':>13}  Per-trial entropies")
    print(f"  {'─'*80}")
    for tid in sorted(universal, key=lambda t: -trial_meta[0]["entropy"][t].item()):
        entropies  = [m["entropy"][tid].item() for m in trial_meta]
        mean_ent   = sum(entropies) / len(entropies)
        per_trial  = "  ".join(f"{e:.4f}" for e in entropies)
        print(f"  {tid:<12} {tok_repr(tid):<30} {mean_ent:>13.4f}  [{per_trial}]")
else:
    print("  (none — no single token appears in every trial's top-k)")

# 2. Frequency table across trials
freq: Counter = Counter()
for s in trial_top_ids:
    freq.update(s)

print(f"\nToken frequency across {n_trials} trials (top-{top_k} per trial):")
print(f"  {'Token ID':<12} {'Token repr':<30} {'Trials':>6}  Bar")
print(f"  {'─'*64}")
for tid, cnt in freq.most_common():
    bar = "█" * cnt + "░" * (n_trials - cnt)
    print(f"  {tid:<12} {tok_repr(tid):<30} {cnt:>6}  {bar}")

# 3. Pairwise Jaccard
print(f"\nPairwise Jaccard similarity of top-{top_k} sets:")
print(f"  {'Pair':<20} {'Jaccard':>8}  Shared tokens")
print(f"  {'─'*64}")
for i in range(n_trials):
    for j in range(i + 1, n_trials):
        a, b    = trial_top_ids[i], trial_top_ids[j]
        jaccard = len(a & b) / len(a | b)
        shared  = [tok_repr(t) for t in sorted(a & b)]
        pair    = f"Trial {i+1} vs {j+1}"
        print(f"  {pair:<20} {jaccard:>8.3f}  {shared}")

# 4. Mean Jaccard across all pairs
jaccards = []
for i in range(n_trials):
    for j in range(i + 1, n_trials):
        a, b = trial_top_ids[i], trial_top_ids[j]
        jaccards.append(len(a & b) / len(a | b))
mean_jaccard = sum(jaccards) / len(jaccards) if jaccards else 1.0

# 5. Verdict
print(f"\n{'='*64}")
print(f"Mean pairwise Jaccard: {mean_jaccard:.3f}")
top1_per_trial = [m["top_ids"][0] for m in trial_meta]
top1_stable    = len(set(top1_per_trial)) == 1

if len(universal) >= 1 and mean_jaccard >= 0.8:
    print(f"✅ STABLE: {len(universal)} token(s) appear in every trial's top-{top_k} "
            f"and mean Jaccard={mean_jaccard:.3f}.")
elif top1_stable:
    tok = tok_repr(top1_per_trial[0])
    print(f"⚠️  PARTIALLY STABLE: The #1 token {tok} is consistent across all trials, "
            f"but the full top-{top_k} set varies (mean Jaccard={mean_jaccard:.3f}).")
else:
    top1_reprs = [tok_repr(t) for t in top1_per_trial]
    print(f"❌ UNSTABLE: Even the #1 token varies across trials. "
            f"Top-1 per trial: {top1_reprs}  (mean Jaccard={mean_jaccard:.3f}).")
print(f"{'='*64}\n")

Loading trial files from: ../outputs/entropy  (glob='*.pt')
  Loaded avg_token_entropy_trial1.pt  (vocab_size=248077)
  Loaded avg_token_entropy_trial2.pt  (vocab_size=248077)
  Loaded avg_token_entropy_trial3.pt  (vocab_size=248077)

3 trial file(s) found.

────────────────────────────────────────────────────────────────
Trial 1: avg_token_entropy_trial1.pt
  Rank   Token ID     Token repr                      Avg Entropy  Count
  ────────────────────────────────────────────────────────────
  1      11346        'Ġdetailed'                          2.3017  10061
  2      1156         'Ġuser'                              2.1853  10240
  3      12121        'ĠSolution'                          2.0627  6252
  4      5764         'ĠResponse'                          2.0300  1772
  5      1179         'Ġthen'                              1.9966  10932
  6      1218         'ful'                                1.9910  10019
  7      4252         'Ġcorrect'                           1.9887  